# XML Annotation Evaluation — GerDraCor TEI Files

This notebook compares **gold-standard** XML annotation files against **automatically annotated** (predicted) XML files, and reports how well the predictions match the gold standard.

Files with the same filename in both folders are paired and evaluated together. Per-file results are shown individually, and a **macro-averaged summary** across all files is displayed at the end.

> **Prerequisite:** run `01_postprocessing.ipynb` on your prediction folder first. This notebook expects clean, postprocessed predictions as input — it does not modify or clean the XML files itself.

All GerDraCor files follow the [TEI P5](https://tei-c.org/guidelines/p5/) schema. Sound events are encoded with two custom inline tags that carry **no namespace**:

| Tag | Meaning |
|---|---|
| `<character_sound>` | A sound produced by a character (speech act, vocal expression, etc.) |
| `<ambient_sound>` | A non-character environmental sound (wind, bells, animals, etc.) |

## Quick Reference — What Do These Metrics Mean?

This notebook reports several different metrics because there are several different ways to ask "did the model get it right?" — from the loosest (did it notice *something* in roughly the right place?) to the strictest (did it get the exact boundaries and the exact label?). Read top to bottom for increasingly strict criteria.

| # | Metric | Plain-language question it answers |
|---|---|---|
| 1 | **Token Accuracy / Token F1** | Looking at the text character by character, how many characters did the model correctly mark as sound vs. not-sound? |
| 2 | **Span Detection (any overlap)** | Did the model notice a sound event *somewhere* in roughly the right place, even if the boundaries don't match exactly and the label (ambient vs. character) is ignored? |
| 3 | **Span Detection (IoU ≥ 0.5)** | Same as above, but now at least half of the predicted span must actually overlap with the gold span — a stricter "roughly the right place." |
| 4 | **Span Classification (any overlap / IoU ≥ 0.5)** | Same two checks as above, but now the predicted label (`ambient_sound` vs. `character_sound`) must also be correct. |
| 5 | **E-F1 (exact span + label)** | The strictest check: the predicted span must start and end at exactly the same character as the gold span, *and* carry the correct label. |
| 6 | **Gamma (Mathet et al.)** | A single agreement score (like Cohen's Kappa, but for spans) that rewards close boundary matches and penalises sloppy ones — instead of a hard yes/no per span. |

**Precision** = of everything the model predicted, how much was correct? **Recall** = of everything that should have been found, how much did the model find? **F1** is the balance between the two.

As you move down the table, scores naturally get lower — this is expected and not a sign that the model is failing. The token-level and "any overlap" numbers tell you whether the model is generally looking in the right places; the E-F1 and Gamma numbers tell you how precisely it draws the boundaries.

## 0. Setup

In [631]:
# Install pygamma-agreement for the Gamma metric.
# Comment out if already installed.
#!pip install pygamma-agreement --quiet

import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
pd.set_option('display.notebook_repr_html', True)
from IPython.display import display, HTML
import re
from pathlib import Path
import warnings
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


## 1. Configuration

Set the paths to the gold-standard folder and the (already postprocessed) prediction folder.

In [632]:
# ── CONFIGURE PATHS HERE ──────────────────────────────────────────────────────

GOLD_DIR = '/Users/sguhr/Desktop/Prediction_vs_test_Sound_in_GerDraCor/20260720_Predictions_and_Testsets_augmented_experiments/20260720_for_prediction_2-5_augmented/20260720_testset_2-5/20260720_testset_2-5'
    #'/Users/sguhr/Desktop/Prediction_vs_test_Sound_in_GerDraCor/20260717_Predictions_and_Testsets_original_plays_experiments/Exp_prose_original_plays/original_plays_testset' # original plays

PRED_DIR = '/Users/sguhr/Desktop/Prediction_vs_test_Sound_in_GerDraCor/20260720_Predictions_and_Testsets_augmented_experiments/20260720_all_prose_model_predicted_plays_augmented/exp_all_prose_predicted_augmented_postprocessed'#_postprocessed'
#'/Users/sguhr/Desktop/Sound_in_Drama/prediction_output_for_evaluation_before_postprocessing/202606_exp3_drama_16prose_output/20260420_exp3_drama_16prose_f3' #pre-postprocessing f1 exp1

# Sound tag names present in the files
SOUND_TAGS = {'ambient_sound', 'character_sound'}

In [633]:
# ── OUTPUT LOGGING SETUP ─────────────────────────────────────────────
# Run this once, near the top. It records everything printed by later
# cells into a buffer (written to disk by the final cell) WITHOUT
# reassigning sys.stdout — so PyCharm/Jupyter keep capturing and storing
# each cell's output normally. This is the fix for outputs disappearing
# on reload.
import builtins, io, sys
from pathlib import Path
from datetime import datetime   # used by the final SAVE ALL OUTPUT cell

_output_log = io.StringIO()

# Idempotent: re-running this cell will not stack the wrapper.
if not getattr(builtins.print, '_tees_to_log', False):
    _real_print = builtins.print

    def _logging_print(*args, **kwargs):
        _real_print(*args, **kwargs)  # -> notebook (captured normally)
        if kwargs.get('file', sys.stdout) in (sys.stdout, None):
            log_kwargs = {k: v for k, v in kwargs.items() if k != 'file'}
            _real_print(*args, file=_output_log, **log_kwargs)  # -> log buffer

    _logging_print._tees_to_log = True
    builtins.print = _logging_print

print('Output logging enabled (notebook capture preserved).')

Output logging enabled (notebook capture preserved).


## 2. Data Structures and XML Parsing

### 2.1 Annotation dataclass

Each detected sound annotation is stored as an `Annotation` object with character-level offsets into the extracted plain text of the play.

In [634]:
@dataclass
class Annotation:
    """Represents a single sound annotation span."""
    start_char: int          # character offset (start) in the reconstructed plain text
    end_char: int            # character offset (end, exclusive)
    label: str               # 'ambient_sound' or 'character_sound'
    text: str = ''           # the literal annotated text
    scene: Optional[str] = None   # scene identifier (e.g. 'act1_scene2')
    parent_tag: Optional[str] = None  # immediate parent element tag (e.g. 'stage', 'p', 'speaker')

    def __repr__(self):
        return (
            f"Annotation(label={self.label!r}, scene={self.scene!r}, "
            f"chars=[{self.start_char},{self.end_char}], "
            f"parent={self.parent_tag!r}, text={self.text[:50]!r})"
        )

### 2.2 XML parser

The parser walks the TEI `<body>` element, reconstructs the plain text of the play, and records character offsets for every `<ambient_sound>` and `<character_sound>` element it encounters.

**Design decisions:**
- **Namespace-agnostic**: the TEI namespace (`http://www.tei-c.org/ns/1.0`) is detected automatically from the root element and used when looking up standard TEI tags (`body`, `div`, `sp`, etc.). Sound tags (`character_sound`, `ambient_sound`) never carry a namespace and are matched by local name only.
- **Scene ID assignment**: computed positionally (`act1_scene2`) since GerDraCor `<div>` elements carry no `n` attribute. Flat corpora (no act level) receive IDs like `scene1`.
- The `parent_tag` field is stored so that annotations inside `<speaker>` elements can be identified and optionally filtered.
- Text inside `<teiHeader>` and `<front>` is ignored; only `<body>` content is processed.

In [635]:
# ── Namespace utilities ───────────────────────────────────────────────────

def _detect_tei_ns(root) -> str:
    """
    Return the TEI namespace URI if the root element declares one,
    otherwise return an empty string.

    GerDraCor files come in two flavours:
      - With namespace : <TEI xmlns="http://www.tei-c.org/ns/1.0" ...>
      - Without        : <TEI xml:id="...">
    """
    tag = root.tag  # e.g. '{http://www.tei-c.org/ns/1.0}TEI'  or just 'TEI'
    if tag.startswith('{'):
        return tag[1:tag.index('}')]   # extract URI between { and }
    return ''


def _local(tag: str) -> str:
    """Strip any namespace prefix from an element tag."""
    return tag.split('}', 1)[1] if '}' in tag else tag


def _tei(local_name: str, ns: str) -> str:
    """
    Return the fully qualified tag name for a standard TEI element,
    prefixed with the namespace if one is present.

    Sound tags (character_sound, ambient_sound) are custom and
    NEVER namespaced — always look them up by plain local name.
    """
    return f'{{{ns}}}{local_name}' if ns else local_name


# ── Recursive text/annotation collector ───────────────────────────────────

def _collect_text_and_annotations(
    elem,
    text_parts: list,
    char_pos: list,
    annotations: list,
    scene_id: str,
    parent_tag: str,
):
    """
    Recursively walk *elem*, collecting all text into *text_parts* and
    recording Annotation objects for every sound element found.

    Parameters
    ----------
    elem       : current XML element
    text_parts : accumulator list for plain-text fragments
    char_pos   : mutable single-element list [current_char_offset]
    annotations: accumulator list for Annotation objects
    scene_id   : scene identifier string (passed down from caller)
    parent_tag : local tag name of *elem*'s parent (for speaker filtering)
    """
    local = _local(elem.tag)
    is_sound = local in SOUND_TAGS

    if is_sound:
        # Record start, collect inner text, store annotation
        ann_start = char_pos[0]
        inner_parts: list = []
        inner_pos = [char_pos[0]]

        if elem.text:
            inner_parts.append(elem.text)
            inner_pos[0] += len(elem.text)

        for child in elem:
            _collect_text_and_annotations(
                child, inner_parts, inner_pos, annotations, scene_id, local
            )
            if child.tail:
                inner_parts.append(child.tail)
                inner_pos[0] += len(child.tail)

        ann_end = inner_pos[0]
        text_parts.extend(inner_parts)
        char_pos[0] = ann_end

        annotations.append(Annotation(
            start_char=ann_start,
            end_char=ann_end,
            label=local,
            text=''.join(inner_parts),
            scene=scene_id,
            parent_tag=parent_tag,
        ))
        return  # children already consumed above

    # Non-sound element: collect text and recurse
    if elem.text:
        text_parts.append(elem.text)
        char_pos[0] += len(elem.text)

    for child in elem:
        _collect_text_and_annotations(
            child, text_parts, char_pos, annotations, scene_id, local
        )
        if child.tail:
            text_parts.append(child.tail)
            char_pos[0] += len(child.tail)


# ── Main parser ────────────────────────────────────────────────────────────

def parse_xml_file(
    xml_path: str,
    exclude_speaker: bool = False,
) -> Tuple[str, List[Annotation]]:
    """
    Parse any GerDraCor TEI XML file and return (plain_text, annotations).

    Handles all known structural variants:
      - With or without TEI namespace declaration.
      - body → div[act] → div[scene]  (multi-act plays, e.g. Wedekind)
      - body → div[scene]             (single-scene Fastnachtsspiele, e.g. Hans Sachs)
      - Any other div type under body is traversed as 'body_level'.

    Sound tags (character_sound, ambient_sound) are always matched by
    local name only — they carry no namespace in any GerDraCor file.

    Parameters
    ----------
    xml_path        : path to the XML file
    exclude_speaker : drop annotations whose parent element is <speaker>

    Returns
    -------
    plain_text  : full reconstructed plain text of the play body
    annotations : list of Annotation objects with character offsets
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    ns = _detect_tei_ns(root)

    # Locate <body> — with or without namespace
    body = root.find(f'.//{_tei("body", ns)}')
    if body is None:
        raise ValueError(
            f'No <body> element found in {xml_path}\n'
            f'  Detected namespace: {ns!r}\n'
            f'  Root tag: {root.tag!r}'
        )

    text_parts: list = []
    annotations: list = []
    char_pos = [0]

    # Leading text of body (rare but possible)
    if body.text:
        text_parts.append(body.text)
        char_pos[0] += len(body.text)

    act_counter = 0
    scene_counter_global = 0  # used for flat (no-act) plays

    for body_child in body:
        bc_local = _local(body_child.tag)
        bc_type  = body_child.get('type', '')

        if bc_local == 'div' and bc_type == 'act':
            # ── Multi-act structure (e.g. Wedekind) ──────────────────────
            act_counter += 1
            scene_counter = 0

            if body_child.text:
                text_parts.append(body_child.text)
                char_pos[0] += len(body_child.text)

            for act_child in body_child:
                ac_local = _local(act_child.tag)
                ac_type  = act_child.get('type', '')

                if ac_local == 'div' and ac_type == 'scene':
                    scene_counter += 1
                    scene_id = f'act{act_counter}_scene{scene_counter}'

                    if act_child.text:
                        text_parts.append(act_child.text)
                        char_pos[0] += len(act_child.text)

                    for scene_child in act_child:
                        _collect_text_and_annotations(
                            scene_child, text_parts, char_pos,
                            annotations, scene_id, ac_local
                        )
                        if scene_child.tail:
                            text_parts.append(scene_child.tail)
                            char_pos[0] += len(scene_child.tail)

                    if act_child.tail:
                        text_parts.append(act_child.tail)
                        char_pos[0] += len(act_child.tail)

                else:
                    # Non-scene child of act (e.g. act-level stage direction)
                    _collect_text_and_annotations(
                        act_child, text_parts, char_pos,
                        annotations, f'act{act_counter}', bc_local
                    )
                    if act_child.tail:
                        text_parts.append(act_child.tail)
                        char_pos[0] += len(act_child.tail)

            if body_child.tail:
                text_parts.append(body_child.tail)
                char_pos[0] += len(body_child.tail)

        elif bc_local == 'div' and bc_type == 'scene':
            # ── Flat structure: scene directly under body (e.g. Hans Sachs) ─
            scene_counter_global += 1
            scene_id = f'scene{scene_counter_global}'

            if body_child.text:
                text_parts.append(body_child.text)
                char_pos[0] += len(body_child.text)

            for scene_child in body_child:
                _collect_text_and_annotations(
                    scene_child, text_parts, char_pos,
                    annotations, scene_id, bc_local
                )
                if scene_child.tail:
                    text_parts.append(scene_child.tail)
                    char_pos[0] += len(scene_child.tail)

            if body_child.tail:
                text_parts.append(body_child.tail)
                char_pos[0] += len(body_child.tail)

        else:
            # ── Any other body-level div (e.g. Dramatis_Personae, front) ──
            _collect_text_and_annotations(
                body_child, text_parts, char_pos,
                annotations, 'body_level', 'body'
            )
            if body_child.tail:
                text_parts.append(body_child.tail)
                char_pos[0] += len(body_child.tail)

    plain_text = ''.join(text_parts)

    if exclude_speaker:
        annotations = [a for a in annotations if a.parent_tag != 'speaker']

    return plain_text, annotations


print('Parser defined — supports namespaced and non-namespaced GerDraCor files.')

Parser defined — supports namespaced and non-namespaced GerDraCor files.


## 3. Load Files

Both gold and prediction files are parsed the same way — there is no special-case handling here, because speaker-name false positives and other known issues have already been removed by the postprocessing notebook.

In [636]:
import os

# Discover paired files: same filename must exist in both folders
gold_files = {f for f in os.listdir(GOLD_DIR) if f.endswith('.xml')}
pred_files = {f for f in os.listdir(PRED_DIR) if f.endswith('.xml')}
paired_files = sorted(gold_files & pred_files)
only_gold = gold_files - pred_files
only_pred = pred_files - gold_files

print(f'Paired files found : {len(paired_files)}')
if only_gold:
    print(f'  Only in gold  (no prediction) : {sorted(only_gold)}')
if only_pred:
    print(f'  Only in pred  (no gold)       : {sorted(only_pred)}')
print()

# Load all paired files and run the sanity check for each
file_data = {}   # filename -> {'gold_text', 'gold_anns', 'pred_text', 'pred_anns'}

for fname in paired_files:
    gold_path = os.path.join(GOLD_DIR, fname)
    pred_path = os.path.join(PRED_DIR, fname)
    gold_text, gold_anns = parse_xml_file(gold_path)
    pred_text, pred_anns = parse_xml_file(pred_path)
    file_data[fname] = dict(
        gold_text=gold_text, gold_anns=gold_anns,
        pred_text=pred_text, pred_anns=pred_anns,
    )

    # Per-file sanity check: gold and prediction should describe the same underlying text
    len_diff = abs(len(gold_text) - len(pred_text))
    if len_diff == 0:
        status = '✓ identical'
    elif len_diff < 200:
        status = f'⚠ differ by {len_diff} chars (likely whitespace)'
    else:
        status = f'✗ DIFFER BY {len_diff} chars — investigate!'
    print(f'{fname:60s}  gold={len(gold_anns):3d} anns  pred={len(pred_anns):3d} anns  text: {status}')

print(f'\nLoaded {len(file_data)} file pair(s).')

Paired files found : 12

ayrer-fassnachtspil-wie-einem-weib-jhr-eygener-mann.xml       gold= 88 anns  pred= 67 anns  text: ✓ identical
borchert-draussen-vor-der-tuer.xml                            gold=355 anns  pred=269 anns  text: ✓ identical
chezy-der-neue-narziss.xml                                    gold= 62 anns  pred= 40 anns  text: ✓ identical
dohm-ein-schuss-ins-schwarze.xml                              gold=177 anns  pred=102 anns  text: ✓ identical
dovsky-mona-lisa.xml                                          gold= 54 anns  pred= 21 anns  text: ✓ identical
ebner-eschenbach-die-veilchen.xml                             gold=131 anns  pred= 72 anns  text: ✓ identical
guenderode-udohla.xml                                         gold= 36 anns  pred= 38 anns  text: ✓ identical
lessing-emilia-galotti.xml                                    gold=202 anns  pred=195 anns  text: ✓ identical
neuber-die-beschuetzte-schauspielkunst.xml                    gold= 34 anns  pred= 20 anns  tex

## 4. Token-level Evaluation

Each character position in the text is assigned a label: `0` (Outside), `1` (ambient_sound), `2` (character_sound). Standard classification metrics (accuracy, precision, recall, F1) are then computed over this character-level sequence.

This gives an intuitive measure of **how many characters were correctly annotated**, weighted by span length.

In [637]:
def annotations_to_char_labels(text_length: int, annotations: List[Annotation]) -> np.ndarray:
    """
    Build a character-level integer label array.
    0 = O (outside), 1 = ambient_sound, 2 = character_sound
    """
    labels = np.zeros(text_length, dtype=int)
    label_map = {'ambient_sound': 1, 'character_sound': 2}
    for ann in annotations:
        lv = label_map.get(ann.label, 0)
        labels[ann.start_char:ann.end_char] = lv
    return labels


def token_level_metrics(
    gold_labels: np.ndarray, pred_labels: np.ndarray
) -> Dict:
    """Compute character-level accuracy, precision, recall, F1."""
    acc = accuracy_score(gold_labels, pred_labels)
    p, r, f, support = precision_recall_fscore_support(
        gold_labels, pred_labels, labels=[1, 2], average=None, zero_division=0
    )
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        gold_labels, pred_labels, labels=[1, 2], average='macro', zero_division=0
    )
    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
        gold_labels, pred_labels, labels=[1, 2], average='micro', zero_division=0
    )
    return {
        'accuracy': acc,
        'per_class': {
            'ambient_sound':   dict(P=p[0], R=r[0], F1=f[0], support=int(support[0])),
            'character_sound': dict(P=p[1], R=r[1], F1=f[1], support=int(support[1])),
        },
        'macro': dict(P=p_macro, R=r_macro, F1=f_macro),
        'micro': dict(P=p_micro, R=r_micro, F1=f_micro),
    }


# Run token-level metrics for every file pair
token_results_all = {}
for fname, d in file_data.items():
    n = len(d['gold_text'])
    gold_labels = annotations_to_char_labels(n, d['gold_anns'])
    pred_labels_raw = annotations_to_char_labels(len(d['pred_text']), d['pred_anns'])
    pred_labels = np.zeros(n, dtype=int)
    m = min(n, len(pred_labels_raw))
    pred_labels[:m] = pred_labels_raw[:m]
    token_results_all[fname] = token_level_metrics(gold_labels, pred_labels)
    # store aligned labels for later re-use
    d['gold_labels'] = gold_labels
    d['pred_labels'] = pred_labels

print('Token-level metrics computed for', len(token_results_all), 'file(s).')

Token-level metrics computed for 12 file(s).


## 5. Span-level Evaluation — Detection

Here the unit of evaluation is a whole annotation span rather than individual characters. A predicted span is a **True Positive** if it overlaps (or exceeds an IoU threshold) with a gold span, regardless of label.

Two thresholds are evaluated:
- **Any overlap** (lenient): even a single shared character counts as a match.
- **IoU ≥ 0.5** (standard): the Intersection over Union of the two spans must be at least 50 %.

In [638]:
def iou(a: Annotation, b: Annotation) -> float:
    """Intersection-over-Union for two character spans."""
    inter_start = max(a.start_char, b.start_char)
    inter_end   = min(a.end_char,   b.end_char)
    inter = max(0, inter_end - inter_start)
    union = (a.end_char - a.start_char) + (b.end_char - b.start_char) - inter
    return inter / union if union > 0 else 0.0


def span_metrics(
    gold: List[Annotation],
    pred: List[Annotation],
    iou_threshold: float = 0.0,
    require_label_match: bool = False,
    label: Optional[str] = None,
) -> Dict:
    """
    Compute span-level Precision, Recall, F1.

    Parameters
    ----------
    iou_threshold       : 0.0 = any overlap; 0.5 = standard IoU threshold
    require_label_match : if True, predicted and gold labels must agree
    label               : restrict evaluation to a single class; None = all
    """
    g = [a for a in gold if label is None or a.label == label]
    p = [a for a in pred if label is None or a.label == label]

    matched_gold = set()
    matched_pred = set()

    for pi, pa in enumerate(p):
        for gi, ga in enumerate(g):
            if gi in matched_gold:
                continue
            label_ok = (not require_label_match) or (pa.label == ga.label)
            if label_ok and iou(pa, ga) > iou_threshold:
                matched_pred.add(pi)
                matched_gold.add(gi)
                break

    tp = len(matched_pred)
    fp = len(p) - tp
    fn = len(g) - len(matched_gold)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return dict(TP=tp, FP=fp, FN=fn, P=precision, R=recall, F1=f1)


# Run span detection for every file pair
span_det_all = {}   # fname -> {det_all, det_iou, per_label_overlap}
for fname, d in file_data.items():
    ga, pa = d['gold_anns'], d['pred_anns']
    span_det_all[fname] = {
        'det_all':  span_metrics(ga, pa, iou_threshold=0.0, require_label_match=False),
        'det_iou':  span_metrics(ga, pa, iou_threshold=0.5, require_label_match=False),
        'per_label': {
            lbl: span_metrics(ga, pa, iou_threshold=0.0, require_label_match=False, label=lbl)
            for lbl in ['ambient_sound', 'character_sound']
        },
    }

print('Span detection metrics computed for', len(span_det_all), 'file(s).')

Span detection metrics computed for 12 file(s).


## 6. Span-level Evaluation — Classification

Same as detection, but a match is only counted when the **label also matches** (`ambient_sound` vs. `character_sound`). This decouples span-finding ability from labelling ability.

In [639]:
# Run span classification for every file pair
span_cls_all = {}
for fname, d in file_data.items():
    ga, pa = d['gold_anns'], d['pred_anns']
    span_cls_all[fname] = {
        'cls_all': span_metrics(ga, pa, iou_threshold=0.0, require_label_match=True),
        'cls_iou': span_metrics(ga, pa, iou_threshold=0.5, require_label_match=True),
        'per_label': {
            lbl: span_metrics(ga, pa, iou_threshold=0.0, require_label_match=True, label=lbl)
            for lbl in ['ambient_sound', 'character_sound']
        },
    }

print('Span classification metrics computed for', len(span_cls_all), 'file(s).')

Span classification metrics computed for 12 file(s).


## 7. Entity-level F1 (E-F1)

The strictest measure: both the **exact character span** and the **label** must match identically. This reflects annotation quality from the perspective of downstream systems that depend on precise boundaries.

In [640]:
def entity_f1(
    gold: List[Annotation],
    pred: List[Annotation],
    label: Optional[str] = None,
) -> Dict:
    """Strict entity-level F1: exact (start, end, label) triple must match."""
    def to_key(a: Annotation):
        return (a.start_char, a.end_char, a.label)

    g = {to_key(a) for a in gold if label is None or a.label == label}
    p = {to_key(a) for a in pred if label is None or a.label == label}

    tp = len(g & p)
    fp = len(p - g)
    fn = len(g - p)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return dict(TP=tp, FP=fp, FN=fn, P=precision, R=recall, E_F1=f1)


# Run E-F1 for every file pair
ef1_results_all = {}
for fname, d in file_data.items():
    ga, pa = d['gold_anns'], d['pred_anns']
    ef1_results_all[fname] = {
        'all':   entity_f1(ga, pa),
        'per_label': {lbl: entity_f1(ga, pa, label=lbl) for lbl in ['ambient_sound', 'character_sound']},
    }

print('E-F1 computed for', len(ef1_results_all), 'file(s).')

E-F1 computed for 12 file(s).


## 8. Mathet et al.'s Gamma (γ)

Gamma is a **unitizing agreement coefficient** that simultaneously accounts for segmentation and categorisation, normalised for chance. Unlike F1-based measures, it penalises boundary imprecision in a principled, continuous way.

Reference: Mathet, Widlöcher & Métivier (2015). *The Unified and Holistic Method Gamma for Inter-Annotator Agreement Measure and Alignment*. Computational Linguistics 41(3).

We use the **pygamma-agreement** library which implements the official algorithm.

In [641]:
import logging
logging.getLogger().setLevel(logging.ERROR)

try:
    from pygamma_agreement import Continuum, CombinedCategoricalDissimilarity
    from pyannote.core import Segment
    import inspect
    PYGAMMA_AVAILABLE = True
except ImportError:
    PYGAMMA_AVAILABLE = False
    print('pygamma-agreement not installed. Falling back to approximation.')


def _make_dissimilarity():
    sig = inspect.signature(CombinedCategoricalDissimilarity.__init__)
    param_names = list(sig.parameters.keys())
    if 'categories' in param_names or 'n_categories' in param_names:
        return CombinedCategoricalDissimilarity(
            categories=list(SOUND_TAGS),
            delta_empty=1.0, alpha=3, beta=1,
        )
    else:
        return CombinedCategoricalDissimilarity(delta_empty=1.0, alpha=3, beta=1)


def compute_gamma_pygamma(
    gold: List[Annotation], pred: List[Annotation]
) -> float:
    """Compute Mathet et al.'s gamma via pygamma-agreement."""
    continuum = Continuum()
    for ann in gold:
        continuum.add('gold', Segment(ann.start_char, ann.end_char), ann.label)
    for ann in pred:
        continuum.add('pred', Segment(ann.start_char, ann.end_char), ann.label)
    dissim = _make_dissimilarity()
    gamma_results = continuum.compute_gamma(dissim, n_samples=30)
    return gamma_results.gamma


def compute_gamma_approx(
    gold: List[Annotation], pred: List[Annotation], text_length: int
) -> float:
    """Approximation of gamma when pygamma-agreement is unavailable."""
    import random

    def disorder(anns_a, anns_b):
        total = 0.0
        matched_b = set()
        for a in anns_a:
            best_d = float('inf')
            best_i = None
            for i, b in enumerate(anns_b):
                pos_d = (abs(a.start_char - b.start_char) + abs(a.end_char - b.end_char))
                cat_d = 0 if a.label == b.label else 1
                d = pos_d / max(text_length, 1) + cat_d
                if d < best_d:
                    best_d, best_i = d, i
            if best_i is not None:
                total += best_d
                matched_b.add(best_i)
            else:
                total += 1.0
        total += len(anns_b) - len(matched_b)
        return total

    observed = disorder(gold, pred)
    rng = random.Random(42)
    all_labels = [a.label for a in gold] + [a.label for a in pred]
    expected_list = []
    for _ in range(100):
        shuffled = rng.sample(all_labels, len(all_labels))
        fake_pred = [
            Annotation(a.start_char, a.end_char, l)
            for a, l in zip(pred, shuffled[:len(pred)])
        ]
        expected_list.append(disorder(gold, fake_pred))
    expected = np.mean(expected_list)
    return 1.0 - (observed / expected) if expected > 0 else 0.0


# Compute gamma for every file pair
gamma_all = {}
for fname, d in file_data.items():
    if PYGAMMA_AVAILABLE:
        g = compute_gamma_pygamma(d['gold_anns'], d['pred_anns'])
    else:
        g = compute_gamma_approx(d['gold_anns'], d['pred_anns'], len(d['gold_text']))
    gamma_all[fname] = g
    print(f'  {fname}: gamma={g:.4f}')

print(f"\nGamma computed for {len(gamma_all)} file(s). Method: {'pygamma-agreement' if PYGAMMA_AVAILABLE else 'approximation'}")

  ayrer-fassnachtspil-wie-einem-weib-jhr-eygener-mann.xml: gamma=0.7425
GLPK Integer Optimizer 5.0
1199 rows, 614 columns, 1258 non-zeros
614 integer variables, all of which are binary
Preprocessing...
1 hidden covering inequaliti(es) were detected
58 rows, 44 columns, 118 non-zeros
44 integer variables, all of which are binary
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 58
Solving LP relaxation...
GLPK Simplex Optimizer 5.0
58 rows, 44 columns, 118 non-zeros
      0: obj =   5.700000000e+02 inf =   2.900e+01 (29)
     16: obj =   5.852488997e+02 inf =   0.000e+00 (0)
*    39: obj =   5.852488997e+02 inf =   0.000e+00 (0)
OPTIMAL LP SOLUTION FOUND
Integer optimization begins...
Long-step dual simplex will be used
+    39: mip =     not found yet >=              -inf        (1; 0)
+    39: >>>>>   5.852488997e+02 >=   5.852488997e+02   0.0% (1; 0)
+    39: mip

## 9. Summary Table

In [642]:
def _build_rows(fname, tok, det, cls, ef1r, gamma):
    """Build summary rows for one file."""
    rows = []
    rows.append({'File': fname, 'Metric': 'Token Accuracy (all chars)',
                 'Precision': '-', 'Recall': '-',
                 'F1': f"{tok['accuracy']:.4f}", 'Note': 'char-level'})
    for lbl in ['ambient_sound', 'character_sound']:
        pc = tok['per_class'][lbl]
        rows.append({'File': fname, 'Metric': f'Token F1 [{lbl}]',
                     'Precision': f"{pc['P']:.4f}", 'Recall': f"{pc['R']:.4f}",
                     'F1': f"{pc['F1']:.4f}", 'Note': 'char-level'})
    rows.append({'File': fname, 'Metric': 'Token Macro F1',
                 'Precision': f"{tok['macro']['P']:.4f}",
                 'Recall':    f"{tok['macro']['R']:.4f}",
                 'F1':        f"{tok['macro']['F1']:.4f}", 'Note': 'char-level'})
    rows.append({'File': fname, 'Metric': 'Span Detection (any overlap)',
                 'Precision': f"{det['det_all']['P']:.4f}", 'Recall': f"{det['det_all']['R']:.4f}",
                 'F1': f"{det['det_all']['F1']:.4f}", 'Note': 'label ignored'})
    rows.append({'File': fname, 'Metric': 'Span Detection (IoU >= 0.5)',
                 'Precision': f"{det['det_iou']['P']:.4f}", 'Recall': f"{det['det_iou']['R']:.4f}",
                 'F1': f"{det['det_iou']['F1']:.4f}", 'Note': 'label ignored'})
    rows.append({'File': fname, 'Metric': 'Span Classification (any overlap)',
                 'Precision': f"{cls['cls_all']['P']:.4f}", 'Recall': f"{cls['cls_all']['R']:.4f}",
                 'F1': f"{cls['cls_all']['F1']:.4f}", 'Note': 'label must match'})
    rows.append({'File': fname, 'Metric': 'Span Classification (IoU >= 0.5)',
                 'Precision': f"{cls['cls_iou']['P']:.4f}", 'Recall': f"{cls['cls_iou']['R']:.4f}",
                 'F1': f"{cls['cls_iou']['F1']:.4f}", 'Note': 'label must match'})
    rows.append({'File': fname, 'Metric': 'E-F1 (exact span + label)',
                 'Precision': f"{ef1r['all']['P']:.4f}", 'Recall': f"{ef1r['all']['R']:.4f}",
                 'F1': f"{ef1r['all']['E_F1']:.4f}", 'Note': 'strict'})
    for lbl in ['ambient_sound', 'character_sound']:
        r = ef1r['per_label'][lbl]
        rows.append({'File': fname, 'Metric': f'E-F1 [{lbl}]',
                     'Precision': f"{r['P']:.4f}", 'Recall': f"{r['R']:.4f}",
                     'F1': f"{r['E_F1']:.4f}", 'Note': 'strict'})
    rows.append({'File': fname, 'Metric': 'Gamma (Mathet et al.)',
                 'Precision': '-', 'Recall': '-',
                 'F1': f'{gamma:.4f}', 'Note': 'unitizing agreement'})
    return rows


all_rows = []
for fname in paired_files:
    all_rows.extend(_build_rows(
        fname,
        token_results_all[fname],
        span_det_all[fname],
        span_cls_all[fname],
        ef1_results_all[fname],
        gamma_all[fname],
    ))

df_all = pd.DataFrame(all_rows)

# ── Per-file tables ──────────────────────────────────────────────────────────
print('=== PER-FILE SUMMARY ===')
for fname in paired_files:
    df_file = df_all[df_all['File'] == fname].drop(columns='File')
    display(
        df_file.style
        .set_caption(f'Evaluation Summary — {fname}')
        .set_table_styles(
            [{'selector': 'caption', 'props': [('font-size', '1.1em'), ('font-weight', 'bold')]}]
        )
        .hide(axis='index')
    )

# ── Macro-average across all files ──────────────────────────────────────────
print('\n=== MACRO-AVERAGE ACROSS ALL FILES ===')
numeric_df = df_all.copy()
for col in ['Precision', 'Recall', 'F1']:
    numeric_df[col] = pd.to_numeric(numeric_df[col], errors='coerce')

macro_df = (
    numeric_df.groupby('Metric')[['Precision', 'Recall', 'F1']]
    .mean()
    .round(4)
    .reset_index()
)
# Preserve original metric order
metric_order = df_all['Metric'].unique().tolist()
macro_df['_order'] = macro_df['Metric'].map({m: i for i, m in enumerate(metric_order)})
macro_df = macro_df.sort_values('_order').drop(columns='_order')
for col in ['Precision', 'Recall', 'F1']:
    macro_df[col] = macro_df[col].map(lambda x: f'{x:.4f}' if pd.notna(x) else '-')

'''
display(
    macro_df.style
    .set_caption(f'Macro-Average over {len(paired_files)} file(s)')
    .set_table_styles(
        [{'selector': 'caption', 'props': [('font-size', '1.1em'), ('font-weight', 'bold')]}]
    )
    .hide(axis='index')
)'''

print('\n=== MACRO-AVERAGE ACROSS ALL FILES ===')
print(macro_df.to_string(index=False))

=== PER-FILE SUMMARY ===


Metric,Precision,Recall,F1,Note
Token Accuracy (all chars),-,-,0.9675,char-level
Token F1 [ambient_sound],0.0000,0.0000,0.0000,char-level
Token F1 [character_sound],0.7212,0.2950,0.4188,char-level
Token Macro F1,0.3606,0.1475,0.2094,char-level
Span Detection (any overlap),0.9254,0.7045,0.8000,label ignored
Span Detection (IoU >= 0.5),0.6716,0.5114,0.5806,label ignored
Span Classification (any overlap),0.9254,0.7045,0.8000,label must match
Span Classification (IoU >= 0.5),0.6716,0.5114,0.5806,label must match
E-F1 (exact span + label),0.5224,0.3977,0.4516,strict
E-F1 [ambient_sound],0.0000,0.0000,0.0000,strict


Metric,Precision,Recall,F1,Note
Token Accuracy (all chars),-,-,0.9556,char-level
Token F1 [ambient_sound],0.7467,0.2607,0.3865,char-level
Token F1 [character_sound],0.7146,0.5113,0.5961,char-level
Token Macro F1,0.7306,0.3860,0.4913,char-level
Span Detection (any overlap),0.7770,0.5887,0.6699,label ignored
Span Detection (IoU >= 0.5),0.6617,0.5014,0.5705,label ignored
Span Classification (any overlap),0.7100,0.5380,0.6122,label must match
Span Classification (IoU >= 0.5),0.6022,0.4563,0.5192,label must match
E-F1 (exact span + label),0.3420,0.2592,0.2949,strict
E-F1 [ambient_sound],0.3500,0.1810,0.2386,strict


Metric,Precision,Recall,F1,Note
Token Accuracy (all chars),-,-,0.9695,char-level
Token F1 [ambient_sound],0.4967,0.2500,0.3326,char-level
Token F1 [character_sound],0.6156,0.2307,0.3356,char-level
Token Macro F1,0.5561,0.2403,0.3341,char-level
Span Detection (any overlap),0.6250,0.4032,0.4902,label ignored
Span Detection (IoU >= 0.5),0.5250,0.3387,0.4118,label ignored
Span Classification (any overlap),0.6250,0.4032,0.4902,label must match
Span Classification (IoU >= 0.5),0.5000,0.3226,0.3922,label must match
E-F1 (exact span + label),0.4000,0.2581,0.3137,strict
E-F1 [ambient_sound],0.2222,0.1538,0.1818,strict


Metric,Precision,Recall,F1,Note
Token Accuracy (all chars),-,-,0.9596,char-level
Token F1 [ambient_sound],0.3119,0.4277,0.3607,char-level
Token F1 [character_sound],0.7155,0.3600,0.4790,char-level
Token Macro F1,0.5137,0.3939,0.4199,char-level
Span Detection (any overlap),0.6569,0.3785,0.4803,label ignored
Span Detection (IoU >= 0.5),0.5980,0.3446,0.4373,label ignored
Span Classification (any overlap),0.6275,0.3616,0.4588,label must match
Span Classification (IoU >= 0.5),0.5686,0.3277,0.4158,label must match
E-F1 (exact span + label),0.4510,0.2599,0.3297,strict
E-F1 [ambient_sound],0.2500,0.2857,0.2667,strict


...

Metric,Precision,Recall,F1,Note
Token Accuracy (all chars),-,-,0.9834,char-level
Token F1 [ambient_sound],1.0000,0.0493,0.0939,char-level
Token F1 [character_sound],0.5808,0.3674,0.4501,char-level
Token Macro F1,0.7904,0.2083,0.2720,char-level
Span Detection (any overlap),0.5500,0.3235,0.4074,label ignored
Span Detection (IoU >= 0.5),0.4500,0.2647,0.3333,label ignored
Span Classification (any overlap),0.5000,0.2941,0.3704,label must match
Span Classification (IoU >= 0.5),0.4500,0.2647,0.3333,label must match
E-F1 (exact span + label),0.4500,0.2647,0.3333,strict
E-F1 [ambient_sound],0.0000,0.0000,0.0000,strict


Metric,Precision,Recall,F1,Note
Token Accuracy (all chars),-,-,0.9725,char-level
Token F1 [ambient_sound],0.0000,0.0000,0.0000,char-level
Token F1 [character_sound],0.8216,0.4739,0.6011,char-level
Token Macro F1,0.4108,0.2370,0.3006,char-level
Span Detection (any overlap),0.8095,0.5484,0.6538,label ignored
Span Detection (IoU >= 0.5),0.7143,0.4839,0.5769,label ignored
Span Classification (any overlap),0.8095,0.5484,0.6538,label must match
Span Classification (IoU >= 0.5),0.7143,0.4839,0.5769,label must match
E-F1 (exact span + label),0.5714,0.3871,0.4615,strict
E-F1 [ambient_sound],0.0000,0.0000,0.0000,strict


Metric,Precision,Recall,F1,Note
Token Accuracy (all chars),-,-,0.9653,char-level
Token F1 [ambient_sound],0.5172,0.3539,0.4202,char-level
Token F1 [character_sound],0.6893,0.2944,0.4126,char-level
Token Macro F1,0.6032,0.3242,0.4164,char-level
Span Detection (any overlap),0.6789,0.4390,0.5332,label ignored
Span Detection (IoU >= 0.5),0.5859,0.3789,0.4602,label ignored
Span Classification (any overlap),0.6282,0.4062,0.4934,label must match
Span Classification (IoU >= 0.5),0.5352,0.3461,0.4204,label must match
E-F1 (exact span + label),0.4535,0.2933,0.3562,strict
E-F1 [ambient_sound],0.1954,0.1667,0.1799,strict


Metric,Precision,Recall,F1,Note
Token Accuracy (all chars),-,-,0.9853,char-level
Token F1 [ambient_sound],0.8486,0.5037,0.6322,char-level
Token F1 [character_sound],0.5868,0.6007,0.5937,char-level
Token Macro F1,0.7177,0.5522,0.6129,char-level
Span Detection (any overlap),0.6029,0.7523,0.6694,label ignored
Span Detection (IoU >= 0.5),0.5221,0.6514,0.5796,label ignored
Span Classification (any overlap),0.6029,0.7523,0.6694,label must match
Span Classification (IoU >= 0.5),0.5147,0.6422,0.5714,label must match
E-F1 (exact span + label),0.4485,0.5596,0.4980,strict
E-F1 [ambient_sound],0.2500,0.2500,0.2500,strict



=== MACRO-AVERAGE ACROSS ALL FILES ===

=== MACRO-AVERAGE ACROSS ALL FILES ===
                           Metric Precision Recall     F1
       Token Accuracy (all chars)         -      - 0.9724
         Token F1 [ambient_sound]    0.5181 0.2168 0.2790
       Token F1 [character_sound]    0.6919 0.3816 0.4796
                   Token Macro F1    0.6050 0.2992 0.3793
     Span Detection (any overlap)    0.7070 0.5101 0.5800
      Span Detection (IoU >= 0.5)    0.5943 0.4253 0.4852
Span Classification (any overlap)    0.6827 0.4911 0.5590
 Span Classification (IoU >= 0.5)    0.5710 0.4064 0.4646
        E-F1 (exact span + label)    0.4515 0.3245 0.3698
             E-F1 [ambient_sound]    0.1890 0.1243 0.1449
           E-F1 [character_sound]    0.4804 0.3584 0.4004
            Gamma (Mathet et al.)         -      - 0.5119


### How to Read the Summary Table

- **Start with Token Macro F1 and Span Detection (any overlap).** If these are reasonably high, the model is generally finding sound events in the right places.
- **Compare Span Detection vs. Span Classification at the same threshold.** A large gap means the model finds the right spans but often assigns the wrong label (`ambient_sound` vs. `character_sound`).
- **Compare "any overlap" vs. "IoU ≥ 0.5" vs. E-F1.** A steep drop from lenient to strict scores means the model finds approximately the right region but struggles with precise boundaries — see the Error Analysis section below for concrete examples.
- **Gamma** gives one overall agreement number that already accounts for boundary closeness, so it is a useful single figure to report alongside E-F1 in a paper.

## 10. Error Analysis

This section looks at the actual mistakes the model made, not just the aggregate scores.

- A **false positive** is a span the model predicted that has no overlapping gold annotation with the same label — the model "saw" a sound that isn't there.
- A **false negative** is a gold annotation the model missed entirely — a real sound event the model didn't catch.

In [643]:
def error_analysis(
    gold: List[Annotation],
    pred: List[Annotation],
    fname: str,
    n_show: int = 10,
) -> Tuple[List[Annotation], List[Annotation]]:
    """
    Identify false positives and false negatives for one file pair.
    A predicted span is a false positive if no gold span of the same
    label overlaps it (IoU > 0). A gold span is a false negative if no
    prediction overlaps it with the same label.
    """
    matched_gold = set()
    matched_pred = set()

    for pi, pa in enumerate(pred):
        for gi, ga in enumerate(gold):
            if gi not in matched_gold and pa.label == ga.label and iou(pa, ga) > 0:
                matched_pred.add(pi)
                matched_gold.add(gi)
                break

    false_positives = [pred[i] for i in range(len(pred)) if i not in matched_pred]
    false_negatives = [gold[i] for i in range(len(gold)) if i not in matched_gold]

    print(f'\n--- {fname} ---')
    print(f'  False Positives (predicted but not in gold): {len(false_positives)}')
    for a in false_positives[:n_show]:
        print(f'    {a}')
    if len(false_positives) > n_show:
        print(f'    ... ({len(false_positives) - n_show} more)')

    print(f'  False Negatives (gold but not predicted): {len(false_negatives)}')
    for a in false_negatives[:n_show]:
        print(f'    {a}')
    if len(false_negatives) > n_show:
        print(f'    ... ({len(false_negatives) - n_show} more)')

    return false_positives, false_negatives


print('=== ERROR ANALYSIS ===')
error_data = {}  # fname -> (false_positives, false_negatives)
for fname, d in file_data.items():
    fp, fn = error_analysis(d['gold_anns'], d['pred_anns'], fname)
    error_data[fname] = (fp, fn)

=== ERROR ANALYSIS ===

--- ayrer-fassnachtspil-wie-einem-weib-jhr-eygener-mann.xml ---
  False Positives (predicted but not in gold): 5
    Annotation(label='character_sound', scene='scene1', chars=[12539,12557], parent='l', text='Drumm sagt mir vor')
    Annotation(label='character_sound', scene='scene1', chars=[17238,17242], parent='stage', text='gibt')
    Annotation(label='character_sound', scene='scene1', chars=[20827,20868], parent='stage', text='Sie schlägt mit dem Korb oder einem Pesen')
    Annotation(label='character_sound', scene='scene1', chars=[21138,21158], parent='stage', text='Sie schlägt wider zu')
    Annotation(label='character_sound', scene='scene1', chars=[26820,26854], parent='l', text='Dann sie bat mir mein Mann bekehrt')
  False Negatives (gold but not predicted): 26
    Annotation(label='character_sound', scene='scene1', chars=[1341,1374], parent='l', text='Vnd speiß mein Weib mit Worten ab')
    Annotation(label='character_sound', scene='scene1', chars=[2856,

### 10.1 Error breakdown by scene

Shows which scenes contribute most false positives and false negatives.

In [644]:
from collections import Counter

print('=== ERROR BREAKDOWN BY SCENE ===')
for fname, (false_positives, false_negatives) in error_data.items():
    fp_by_scene = Counter(a.scene for a in false_positives)
    fn_by_scene = Counter(a.scene for a in false_negatives)
    all_scenes = sorted(set(fp_by_scene) | set(fn_by_scene))
    if not all_scenes:
        print(f'\n{fname}: no errors.')
        continue
    scene_df = pd.DataFrame({
        'Scene': all_scenes,
        'False Positives': [fp_by_scene.get(s, 0) for s in all_scenes],
        'False Negatives': [fn_by_scene.get(s, 0) for s in all_scenes],
    })
    scene_df['Total Errors'] = scene_df['False Positives'] + scene_df['False Negatives']
    scene_df = scene_df.sort_values('Total Errors', ascending=False)
    display(
        scene_df.style
        .set_caption(f'Errors by Scene — {fname}')
        .hide(axis='index')
    )

=== ERROR BREAKDOWN BY SCENE ===


Scene,False Positives,False Negatives,Total Errors
scene1,5,26,31


Scene,False Positives,False Negatives,Total Errors
scene7,36,74,110
scene5,23,42,65
scene1,2,20,22
scene6,10,12,22
scene4,2,10,12
scene3,4,5,9
scene2,1,1,2


Scene,False Positives,False Negatives,Total Errors
scene3,3,24,27
scene7,5,6,11
scene2,1,3,4
scene4,4,0,4
scene6,1,2,3
scene1,0,2,2
scene5,1,0,1


Scene,False Positives,False Negatives,Total Errors
scene11,4,31,35
scene9,3,23,26
scene16,3,13,16
scene13,8,6,14
scene5,7,6,13
scene15,2,7,9
scene18,4,5,9
scene2,0,7,7
scene3,3,3,6
scene4,0,6,6


...

Scene,False Positives,False Negatives,Total Errors
act3_scene1,8,5,13
act3_scene8,4,7,11
act4_scene1,3,6,9
act4_scene7,2,6,8
act5_scene5,4,4,8
act4_scene3,3,5,8
act4_scene8,1,6,7
act4_scene5,5,1,6
act1_scene8,3,1,4
act3_scene5,1,2,3


Scene,False Positives,False Negatives,Total Errors
act1_scene10,2,5,7
act1,2,4,6
act1_scene11,0,4,4
act1_scene2,2,2,4
act1_scene8,4,0,4
act1_scene13,0,3,3
act1_scene3,0,2,2
act1_scene5,0,2,2
act1_scene12,0,1,1
act1_scene6,0,1,1


Scene,False Positives,False Negatives,Total Errors
scene1,4,14,18


Scene,False Positives,False Negatives,Total Errors
act2_scene3,25,50,75
act4_scene5,18,46,64
act5_scene1,13,33,46
act1_scene2,17,27,44
act2_scene2,14,29,43
act5_scene2,8,34,42
act3_scene2,12,29,41
act1_scene3,7,14,21
act4_scene3,3,15,18
act4_scene2,4,12,16


Scene,False Positives,False Negatives,Total Errors
act3_scene1,8,3,11
act3_scene3,6,2,8
act3_scene7,4,4,8
act1_scene3,3,4,7
act1_scene5,6,1,7
act3_scene5,3,4,7
act1_scene2,4,2,6
act2_scene1,4,2,6
act2_scene7,4,2,6
act2_scene2,3,2,5


### 10.2 Label confusion

For spans that were matched (any overlap) but received the wrong label, show what the gold label was versus the predicted label.

In [645]:
print('=== LABEL CONFUSION ===')
for fname, d in file_data.items():
    label_confusion = []
    for pa in d['pred_anns']:
        for ga in d['gold_anns']:
            if iou(pa, ga) > 0 and pa.label != ga.label:
                label_confusion.append({
                    'gold_label': ga.label, 'pred_label': pa.label,
                    'gold_text': ga.text[:60], 'pred_text': pa.text[:60],
                    'scene': ga.scene,
                })
    print(f'\n{fname}: {len(label_confusion)} label confusion(s)')
    if label_confusion:
        display(pd.DataFrame(label_confusion).head(20))

=== LABEL CONFUSION ===

ayrer-fassnachtspil-wie-einem-weib-jhr-eygener-mann.xml: 0 label confusion(s)

borchert-draussen-vor-der-tuer.xml: 23 label confusion(s)


,gold_label,pred_label,gold_text,pred_text,scene
0,ambient_sound,character_sound,Die Elbe\n quasselt weiter,Die Elbe\n quasselt weiter,scene1
1,ambient_sound,character_sound,weil jemand so grauenhaft schreit,weil jemand so grauenhaft schreit,scene5
2,ambient_sound,character_sound,was sie\n brüllen,was sie\n brüllen,scene5
3,ambient_sound,character_sound,brüllen sie,brüllen sie,scene5
4,character_sound,ambient_sound,und jede Nacht der furchtbare Schrei,und jede Nacht der furchtbare Schrei,scene5
5,ambient_sound,character_sound,Die fragen jede Nacht,fragen jede Nacht,scene5
6,ambient_sound,character_sound,und fragen,und fragen,scene5
7,ambient_sound,character_sound,Und die flüstern\n dann aus der Dun...,flüstern,scene5
8,ambient_sound,character_sound,Und die flüstern\n dann aus der Dun...,dann aus der Dunkelheit,scene5
9,ambient_sound,character_sound,So flüstern sie,So flüstern sie,scene5



chezy-der-neue-narziss.xml: 1 label confusion(s)


,gold_label,pred_label,gold_text,pred_text,scene
0,character_sound,ambient_sound,Wie bange klopft mein Herz,Wie bange klopft,scene5



dohm-ein-schuss-ins-schwarze.xml: 4 label confusion(s)


...

,gold_label,pred_label,gold_text,pred_text,scene
0,ambient_sound,character_sound,Man läßt der Stimme Laut hell durch die Kehle ...,Man läßt der Stimme,act1_scene2
1,ambient_sound,character_sound,Man läßt der Stimme Laut hell durch die Kehle ...,durch,act1_scene2
2,ambient_sound,character_sound,Der mit so starkem Laut in dieser Gegend schallt,so starkem Laut,act1_scene8
3,ambient_sound,character_sound,Der mit so starkem Laut in dieser Gegend schallt,schallt,act1_scene8



sachs-eulenspiegel-mit-dem-blauen-hosentuch.xml: 0 label confusion(s)

schiller-die-raeuber.xml: 23 label confusion(s)


,gold_label,pred_label,gold_text,pred_text,scene
0,ambient_sound,character_sound,die Beleidigte schreien\n laut um...,Beleidigte schreien\n laut,act1_scene1
1,character_sound,ambient_sound,Auf den Boden stampfend,Auf den Boden stampfend,act1_scene2
2,character_sound,ambient_sound,Schäumend auf die Erde\n stampfend,auf die Erde,act1_scene2
3,character_sound,ambient_sound,mit lärmendem Geschrei,mit lärmendem Geschrei,act1_scene2
4,character_sound,ambient_sound,"gräßlich schreiend, sich die Haare ausraufend",schreiend,act2_scene2
5,character_sound,ambient_sound,"schreiend, sein Gesicht zerfleischend",schreiend,act2_scene2
6,character_sound,ambient_sound,Itzt pfeif ich,Itzt pfeif ich,act2_scene3
7,character_sound,ambient_sound,daß ihm die Zähne klapperten,daß ihm die Zähne klapperten,act2_scene3
8,character_sound,ambient_sound,Trillert ein Liedchen,Trillert ein Liedchen,act2_scene3
9,character_sound,ambient_sound,mitten in den Blumen der glücklichen\n ...,in den Blumen der glücklichen\n W...,act3_scene2



wedekind-fruehlings-erwachen.xml: 1 label confusion(s)


,gold_label,pred_label,gold_text,pred_text,scene
0,ambient_sound,character_sound,Der Wind pfeift auf jedem Stein aus einer ande...,pfeift auf jedem Stein aus einer anderen Tonart,act3_scene7


In [646]:
# ── SAVE ALL OUTPUT ──────────────────────────────────────────────────
# (Nothing to restore: logging uses a print wrapper, not a stdout swap.)

pred_name = Path(PRED_DIR).name          # last folder of the prediction path
gold_name = Path(GOLD_DIR).name          # last folder of the gold path
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')

out_path = Path.home() / 'Downloads' / f'eval_{pred_name}_vs_{gold_name}_{stamp}.txt'

header = (
    f"{'='*60}\n"
    f"Evaluation output log\n"
    f"PRED_DIR : {PRED_DIR}\n"
    f"GOLD_DIR : {GOLD_DIR}\n"
    f"Saved    : {stamp}\n"
    f"{'='*60}\n\n"
)

with open(out_path, 'w') as f:
    f.write(header)
    f.write(_output_log.getvalue())

print(f'Saved evaluation log to: {out_path}')

Saved evaluation log to: /Users/sguhr/Downloads/eval_exp_all_prose_predicted_augmented_postprocessed_vs_20260720_testset_2-5_20260722_162214.txt
